# BlueStone Real Estate - Deployment Ready Notebook

This notebook takes the BlueStone modeling workflow a step further and prepares it for **deployment**.

It includes:
1. Data loading from your local path
2. Training the **Price Prediction** and **Inquiry SLA** models
3. Saving deployment-ready artifacts
4. Generating a **FastAPI inference app** automatically
5. Generating a **requirements.txt** file
6. Generating a **Dockerfile**
7. Running local prediction checks

Dataset path used:
`C:\Users\user\Documents\LML BlueStoneKay\Data\`

## Important compatibility fix
Your environment does not support:

```python
mean_squared_error(y_true, y_pred, squared=False)
```

So this notebook uses:

```python
rmse = mean_squared_error(y_true, y_pred) ** 0.5
```


In [ ]:
import warnings
from pathlib import Path
import json
import textwrap

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)
import joblib

warnings.filterwarnings('ignore')


## 1. Configure Local Paths

In [ ]:
BASE_DIR = Path(r"C:\Users\user\Documents\LML BlueStoneKay\Data")
MODEL_DIR = BASE_DIR / 'models'
DEPLOY_DIR = BASE_DIR / 'deployment_ready'
API_DIR = DEPLOY_DIR / 'api'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
API_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR :', BASE_DIR)
print('MODEL_DIR:', MODEL_DIR)
print('API_DIR  :', API_DIR)


## 2. Load Datasets

In [ ]:
properties = pd.read_csv(BASE_DIR / 'properties_clean.csv')
prices = pd.read_csv(BASE_DIR / 'property_prices_history.csv')
users = pd.read_csv(BASE_DIR / 'users_clean.csv')
inquiries = pd.read_csv(BASE_DIR / 'customer_inquiries_clean.csv')
agents = pd.read_excel(BASE_DIR / 'agents.xlsx')

print('properties:', properties.shape)
print('prices    :', prices.shape)
print('users     :', users.shape)
print('inquiries :', inquiries.shape)
print('agents    :', agents.shape)


## 3. Helper Functions

In [ ]:
def parse_budget_range(df, column='budget_range'):
    out = df.copy()
    parts = out[column].astype(str).str.extract(r'(?P<budget_min>\d+\.?\d*)-(?P<budget_max>\d+\.?\d*)')
    out['budget_min'] = pd.to_numeric(parts['budget_min'], errors='coerce')
    out['budget_max'] = pd.to_numeric(parts['budget_max'], errors='coerce')
    out['budget_mid'] = (out['budget_min'] + out['budget_max']) / 2
    out['budget_missing_flag'] = out['budget_min'].isna().astype(int)
    return out

def build_current_properties(df):
    out = df.copy()
    if 'is_current' in out.columns:
        out = out[out['is_current'] == True].copy()
    if 'data_quality_score' in out.columns:
        out = out.sort_values(['property_id', 'data_quality_score'], ascending=[True, False])
    return out.drop_duplicates('property_id', keep='first')

def build_latest_prices(df):
    out = df.copy()
    out['price_date'] = pd.to_datetime(out['price_date'], errors='coerce')
    for col in ['zestimate', 'assessed_value', 'list_price', 'price_per_sqft', 'price_change_pct']:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    out = out.sort_values(['property_id', 'price_date'], ascending=[True, False])
    out = out.drop_duplicates('property_id', keep='first')
    return out.rename(columns={
        'price_date': 'latest_price_date',
        'list_price': 'latest_list_price',
        'zestimate': 'latest_zestimate',
        'assessed_value': 'latest_assessed_value',
        'price_per_sqft': 'latest_price_per_sqft',
        'price_change_pct': 'latest_price_change_pct',
    })

def build_agent_features(df, as_of_year=2026):
    out = df.copy()
    if 'hire_date' in out.columns:
        out['hire_date'] = pd.to_datetime(out['hire_date'], errors='coerce')
        out['agent_experience_years'] = as_of_year - out['hire_date'].dt.year
    return out

def make_preprocessor(X):
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ])


## 4. Build Curated Input Tables

In [ ]:
properties_current = build_current_properties(properties)
prices_latest = build_latest_prices(prices)
users_feat = parse_budget_range(users)
agents_feat = build_agent_features(agents)

print(properties_current.shape, prices_latest.shape, users_feat.shape, agents_feat.shape)


## 5. Train Deployment Price Model

In [ ]:
price_df = properties_current.merge(prices_latest, on='property_id', how='inner')
price_df['property_age'] = 2026 - price_df['year_built']
price_df['bed_bath_ratio'] = price_df['bedrooms'] / np.maximum(price_df['bathrooms'], 1)
price_df['sqft_per_bedroom'] = price_df['sqft'] / np.maximum(price_df['bedrooms'], 1)
price_df['has_valid_geo'] = (
    price_df['latitude'].between(-90, 90) & price_df['longitude'].between(-180, 180)
).astype(int)

price_model_df = price_df[[
    'property_id','property_type','bedrooms','bathrooms','sqft','lot_size','year_built',
    'listing_status','latitude','longitude','data_quality_score','property_age',
    'bed_bath_ratio','sqft_per_bedroom','has_valid_geo','latest_list_price'
]].copy()

X = price_model_df.drop(columns=['latest_list_price', 'property_id'])
y = price_model_df['latest_list_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

price_model = Pipeline([
    ('preprocessor', make_preprocessor(X)),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1, min_samples_leaf=2))
])
price_model.fit(X_train, y_train)
pred = price_model.predict(X_test)

price_metrics = {
    'rmse': float(mean_squared_error(y_test, pred) ** 0.5),
    'mae': float(mean_absolute_error(y_test, pred)),
    'r2': float(r2_score(y_test, pred))
}
price_metrics


## 6. Train Deployment Inquiry SLA Model

In [ ]:
inq = inquiries.copy()
inq['inquiry_date'] = pd.to_datetime(inq['inquiry_date'], errors='coerce')
inq['response_date'] = pd.to_datetime(inq['response_date'], errors='coerce')
inq['inquiry_hour'] = inq['inquiry_date'].dt.hour
inq['inquiry_dayofweek'] = inq['inquiry_date'].dt.dayofweek
inq['has_property_id'] = inq['property_id'].notna().astype(int)
inq = inq[inq['response_time_hours'].notna()].copy()
inq['sla_breach_24h'] = (inq['response_time_hours'] > 24).astype(int)

sla_df = (
    inq.merge(users_feat[['user_id','user_type','budget_mid','budget_missing_flag']], on='user_id', how='left')
       .merge(properties_current[['property_id','property_type','bedrooms','bathrooms','sqft','listing_status','data_quality_score']], on='property_id', how='left')
       .merge(agents_feat[['agent_id','office_location','status','agent_experience_years']], on='agent_id', how='left', suffixes=('','_agent'))
)

sla_df['property_missing_flag'] = sla_df['property_type'].isna().astype(int)
sla_df = sla_df.rename(columns={'status_agent':'agent_status'})
sla_model_df = sla_df[[
    'inquiry_id','inquiry_type','status','inquiry_hour','inquiry_dayofweek','has_property_id',
    'user_type','budget_mid','budget_missing_flag','property_type','bedrooms','bathrooms',
    'sqft','listing_status','data_quality_score','office_location','agent_status',
    'agent_experience_years','property_missing_flag','sla_breach_24h'
]].copy()

X = sla_model_df.drop(columns=['sla_breach_24h', 'inquiry_id'])
y = sla_model_df['sla_breach_24h']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sla_model = Pipeline([
    ('preprocessor', make_preprocessor(X)),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, min_samples_leaf=2, class_weight='balanced'))
])
sla_model.fit(X_train, y_train)
pred = sla_model.predict(X_test)

sla_metrics = {
    'accuracy': float(accuracy_score(y_test, pred)),
    'precision': float(precision_score(y_test, pred, zero_division=0)),
    'recall': float(recall_score(y_test, pred, zero_division=0)),
    'f1': float(f1_score(y_test, pred, zero_division=0))
}
sla_metrics


## 7. Save Deployment Artifacts

In [ ]:
joblib.dump(price_model, MODEL_DIR / 'price_model.joblib')
joblib.dump(sla_model, MODEL_DIR / 'sla_model.joblib')

with open(MODEL_DIR / 'price_model_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(price_metrics, f, indent=2)

with open(MODEL_DIR / 'sla_model_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(sla_metrics, f, indent=2)

print('Artifacts saved to:', MODEL_DIR)


## 8. Generate FastAPI App Automatically

In [ ]:
api_code = '''
from pathlib import Path
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

BASE_DIR = Path(__file__).resolve().parent.parent
MODEL_DIR = BASE_DIR / 'models'

app = FastAPI(title='BlueStone ML API', version='1.0.0')

class PriceRequest(BaseModel):
    property_type: str
    bedrooms: int
    bathrooms: int
    sqft: float
    lot_size: float
    year_built: int
    listing_status: str
    latitude: float
    longitude: float
    data_quality_score: float

class SLARequest(BaseModel):
    inquiry_type: str
    status: str
    inquiry_hour: int
    inquiry_dayofweek: int
    has_property_id: int
    user_type: str
    budget_mid: float | None = None
    budget_missing_flag: int = 0
    property_type: str | None = None
    bedrooms: int | None = None
    bathrooms: int | None = None
    sqft: float | None = None
    listing_status: str | None = None
    data_quality_score: float | None = None
    office_location: str | None = None
    agent_status: str | None = None
    agent_experience_years: float | None = None
    property_missing_flag: int = 0

price_model = joblib.load(MODEL_DIR / 'price_model.joblib')
sla_model = joblib.load(MODEL_DIR / 'sla_model.joblib')

@app.get('/health')
def health():
    return {'status': 'ok'}

@app.post('/predict/price')
def predict_price(req: PriceRequest):
    payload = pd.DataFrame([{
        'property_type': req.property_type,
        'bedrooms': req.bedrooms,
        'bathrooms': req.bathrooms,
        'sqft': req.sqft,
        'lot_size': req.lot_size,
        'year_built': req.year_built,
        'listing_status': req.listing_status,
        'latitude': req.latitude,
        'longitude': req.longitude,
        'data_quality_score': req.data_quality_score,
        'property_age': 2026 - req.year_built,
        'bed_bath_ratio': req.bedrooms / max(req.bathrooms, 1),
        'sqft_per_bedroom': req.sqft / max(req.bedrooms, 1),
        'has_valid_geo': int(-90 <= req.latitude <= 90 and -180 <= req.longitude <= 180),
    }])
    pred = price_model.predict(payload)[0]
    return {'predicted_list_price': float(pred)}

@app.post('/predict/inquiry-sla')
def predict_sla(req: SLARequest):
    payload = pd.DataFrame([req.model_dump()])
    pred = sla_model.predict(payload)[0]
    prob = None
    estimator = sla_model.named_steps['model']
    if hasattr(estimator, 'predict_proba'):
        prob = float(sla_model.predict_proba(payload)[0][1])
    return {
        'sla_breach_24h_prediction': int(pred),
        'sla_breach_probability': prob
    }
'''

(API_DIR / 'api_app.py').write_text(api_code, encoding='utf-8')
print((API_DIR / 'api_app.py'))


## 9. Generate Requirements File

In [ ]:
requirements_text = '''
pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.1.0
joblib>=1.2.0
fastapi>=0.100.0
uvicorn>=0.22.0
openpyxl>=3.1.0
pydantic>=2.0.0
'''

(DEPLOY_DIR / 'requirements.txt').write_text(requirements_text, encoding='utf-8')
print(DEPLOY_DIR / 'requirements.txt')


## 10. Generate Dockerfile

In [ ]:
dockerfile_text = '''
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000
CMD ["uvicorn", "api.api_app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

(DEPLOY_DIR / 'Dockerfile').write_text(dockerfile_text, encoding='utf-8')
print(DEPLOY_DIR / 'Dockerfile')


## 11. Generate Local Run Guide

In [ ]:
readme_text = '''
# BlueStone Deployment Ready Package

## Files
- models/price_model.joblib
- models/sla_model.joblib
- api/api_app.py
- requirements.txt
- Dockerfile

## Local API Run
```bash
cd deployment_ready
pip install -r requirements.txt
uvicorn api.api_app:app --reload
```

## API Docs
Open:
http://127.0.0.1:8000/docs

## Docker Build
```bash
docker build -t bluestone-ml-api .
docker run -p 8000:8000 bluestone-ml-api
```
'''

(DEPLOY_DIR / 'README.md').write_text(readme_text, encoding='utf-8')
print(DEPLOY_DIR / 'README.md')


## 12. Sample Local Prediction Check

In [ ]:
sample_price_input = price_model_df.drop(columns=['latest_list_price', 'property_id']).iloc[[0]].copy()
sample_sla_input = sla_model_df.drop(columns=['sla_breach_24h', 'inquiry_id']).iloc[[0]].copy()

print('Sample deployed price prediction:', float(price_model.predict(sample_price_input)[0]))
print('Sample deployed SLA prediction :', int(sla_model.predict(sample_sla_input)[0]))


## 13. Deployment Package Output

After running this notebook successfully, you will have:

- trained models in `models/`
- metrics JSON files in `models/`
- `deployment_ready/api/api_app.py`
- `deployment_ready/requirements.txt`
- `deployment_ready/Dockerfile`
- `deployment_ready/README.md`

That means you can move directly to:

```bash
cd C:\Users\user\Documents\LML BlueStoneKay\Data\deployment_ready
pip install -r requirements.txt
uvicorn api.api_app:app --reload
```
